# Ejercicio 5: Espacio Vectorial
**Objetivo de la práctica**
Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus
Carga el corpus y realiza las etapas de preprocesamiento (limpieza, tokenización, stemming).

In [1]:
import pandas as pd
import numpy as np
import re
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

# Para evitar tiempos excesivos de procesamiento en el ejercicio, limitamos el dataset
LIMIT = 1000 

# Carga el corpus
print("Cargando dataset...")
df_corpus = pd.read_csv(
    'wikipedia_text_corpus.csv',
    nrows=LIMIT
)

print(f"Corpus cargado con {len(df_corpus)} documentos.")

display(df_corpus.head())

Cargando dataset...
Corpus cargado con 1000 documentos.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


In [2]:
# Función para procesar cada documento (como en limpieza.ipynb)

stemmer = PorterStemmer()

def process_document(text):
    # Validar que el dato sea texto
    if not isinstance(text, str):
        return ""

    # Eliminar caracteres especiales
    cleaned = re.sub(r'[^\w\s]', ' ', text)

    # Eliminar números
    cleaned = re.sub(r'\d+', '', cleaned)

    # Convertir a minúsculas
    cleaned = cleaned.lower()

    # Tokenización simple
    tokens = cleaned.split()

    # Aplicar stemming
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    # Unir nuevamente como texto
    return ' '.join(stemmed_tokens)


print("Iniciando el preprocesamiento (Limpieza, Tokenización, Stemming)...")
df_corpus['processed'] = df_corpus['text'].apply(process_document)
print("Preprocesamiento completado.")
display(df_corpus[['text', 'processed']].head())

Iniciando el preprocesamiento (Limpieza, Tokenización, Stemming)...


Preprocesamiento completado.


,text,processed
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo formerli a novo is a comput servic...
1,Battery indicator\n\nA battery indicator (also...,batteri indic a batteri indic also known as a ...
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob peas robert allen peas august â â june wa ...
3,CAVNET\n\nCAVNET was a secure military forum w...,cavnet cavnet wa a secur militari forum which ...
4,CLidar\n\nThe CLidar is a scientific instrumen...,clidar the clidar is a scientif instrument use...


## Parte 1: Recuperación con TF-IDF
Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF.

In [3]:
# Crear el vectorizador TF-IDF y transformar el corpus procesado
vectorizer_tfidf = TfidfVectorizer()
tfidf_matrix = vectorizer_tfidf.fit_transform(df_corpus['processed'])

print(f"Matriz TF-IDF creada con forma: {tfidf_matrix.shape}")

Matriz TF-IDF creada con forma: (1000, 25472)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Conjunto de 10 queries
queries = [
    "artificial intelligence computer",
    "history of the world",
    "global warming and climate change",
    "how to play video games",
    "space exploration and mars",
    "electric vehicles energy",
    "united states politics",
    "the immune system health",
    "famous classical music composers",
    "stock market trends economy"
]

def search_tfidf(query, vectorizer, matrix, df, top_k=5):
    query_processed = process_document(query)
    query_vector = vectorizer.transform([query_processed])
    similarities = cosine_similarity(query_vector, matrix)[0]
    
    # Obtener los top_k índices (ordenados de mayor a menor)
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        # Solo agregar si hay algo de similitud
        if similarities[idx] > 0:
            results.append({
                'doc_id': idx,
                'score': similarities[idx],
                'text': df.iloc[idx]['text'][:150].replace('\n', ' ') + "..."
            })
    return results

# Verificar la recuperación del sistema
print("=== Búsqueda usando TF-IDF ===")
for i, q in enumerate(queries):
    print(f"\n--- Query {i+1}: '{q}' ---")
    results = search_tfidf(q, vectorizer_tfidf, tfidf_matrix, df_corpus)
    if not results:
        print("No se encontraron documentos relevantes.")
    for r in results:
        print(f"Doc ID: {r['doc_id']:<4} | Similitud: {r['score']:.4f} | Texto: {r['text']}")

=== Búsqueda usando TF-IDF ===

--- Query 1: 'artificial intelligence computer' ---
Doc ID: 485  | Similitud: 0.3430 | Texto: Strategic Computing Initiative  The United States government's Strategic Computing Initiative funded research into advanced computer hardware and arti...
Doc ID: 669  | Similitud: 0.2897 | Texto: Open Letter on Artificial Intelligence  In January 2015, Stephen Hawking, Elon Musk, and dozens of artificial intelligence experts signed an open lett...
Doc ID: 671  | Similitud: 0.1805 | Texto: Organizational intelligence  Organizational Intelligence (OI) is the capability of an organization to comprehend and conclude knowledge relevant to it...
Doc ID: 996  | Similitud: 0.1594 | Texto: Computer-aided maintenance  Computer-aided maintenance (not to be confused with CAM which usually stands for Computer Aided Manufacturing) refers to s...
Doc ID: 120  | Similitud: 0.1593 | Texto: List of computer books  Computer fundamentals by bikramjit nath    ...

--- Query 2: 'hist

## Parte 2: Recuperación con BM25
Implementa un sistema de recuperación usando el modelo BM25.

In [5]:
# !pip install rank_bm25
from rank_bm25 import BM25Okapi

# BM25 requiere los documentos como listas de tokens (ya procesados)
tokenized_corpus = [doc.split() for doc in df_corpus['processed']]

bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, bm25_model, df, top_k=5):
    query_processed = process_document(query)
    query_tokens = query_processed.split()
    
    doc_scores = bm25_model.get_scores(query_tokens)
    
    # Obtener los top_k índices
    top_indices = doc_scores.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        if doc_scores[idx] > 0:
            results.append({
                'doc_id': idx,
                'score': doc_scores[idx],
                'text': df.iloc[idx]['text'][:150].replace('\n', ' ') + "..."
            })
    return results

# Verificar la recuperación
print("=== Búsqueda usando BM25 ===")
for i, q in enumerate(queries):
    print(f"\n--- Query {i+1}: '{q}' ---")
    results = search_bm25(q, bm25, df_corpus)
    if not results:
        print("No se encontraron documentos relevantes.")
    for r in results:
        print(f"Doc ID: {r['doc_id']:<4} | Score: {r['score']:.4f} | Texto: {r['text']}")

=== Búsqueda usando BM25 ===

--- Query 1: 'artificial intelligence computer' ---
Doc ID: 485  | Score: 15.3722 | Texto: Strategic Computing Initiative  The United States government's Strategic Computing Initiative funded research into advanced computer hardware and arti...
Doc ID: 669  | Score: 14.6849 | Texto: Open Letter on Artificial Intelligence  In January 2015, Stephen Hawking, Elon Musk, and dozens of artificial intelligence experts signed an open lett...
Doc ID: 265  | Score: 12.3963 | Texto: ILabs  iLabs is a non-profit Milan-based organization pursuing multidisciplinary research on radical extension of human life-span. It was founded in 1...
Doc ID: 996  | Score: 11.4551 | Texto: Computer-aided maintenance  Computer-aided maintenance (not to be confused with CAM which usually stands for Computer Aided Manufacturing) refers to s...
Doc ID: 862  | Score: 10.8409 | Texto: Liz Bacon  Liz Bacon (born 27 September 1963) is a Professor of Software Engineering and Deputy Pro Vice-Ch

## Parte 3: Comparación de resultados
Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación.

In [6]:
# Comparamos los resultados de ambos algoritmos
for i, q in enumerate(queries):
    print(f"\n" + "="*50)
    print(f"Query {i+1}: '{q}'")
    print("="*50)
    
    res_tfidf = search_tfidf(q, vectorizer_tfidf, tfidf_matrix, df_corpus, top_k=5)
    res_bm25 = search_bm25(q, bm25, df_corpus, top_k=5)
    
    print("{:<25} | {:<25}".format("Recuperación TF-IDF", "Recuperación BM25"))
    print("-"*55)
    
    max_len = max(len(res_tfidf), len(res_bm25))
    for j in range(max_len):
        str_tfidf = f"Doc {res_tfidf[j]['doc_id']} ({res_tfidf[j]['score']:.4f})" if j < len(res_tfidf) else "-"
        str_bm25 = f"Doc {res_bm25[j]['doc_id']} ({res_bm25[j]['score']:.4f})" if j < len(res_bm25) else "-"
        print("{:<25} | {:<25}".format(str_tfidf, str_bm25))


Query 1: 'artificial intelligence computer'
Recuperación TF-IDF       | Recuperación BM25        
-------------------------------------------------------
Doc 485 (0.3430)          | Doc 485 (15.3722)        
Doc 669 (0.2897)          | Doc 669 (14.6849)        
Doc 671 (0.1805)          | Doc 265 (12.3963)        
Doc 996 (0.1594)          | Doc 996 (11.4551)        
Doc 120 (0.1593)          | Doc 862 (10.8409)        

Query 2: 'history of the world'
Recuperación TF-IDF       | Recuperación BM25        
-------------------------------------------------------
Doc 430 (0.2488)          | Doc 150 (12.3630)        
Doc 338 (0.2423)          | Doc 283 (12.0283)        
Doc 515 (0.2281)          | Doc 363 (11.9869)        
Doc 464 (0.2150)          | Doc 974 (11.9257)        
Doc 915 (0.2027)          | Doc 560 (11.9194)        

Query 3: 'global warming and climate change'
Recuperación TF-IDF       | Recuperación BM25        
-------------------------------------------------------
Doc 94